# HCAD Property Master and Rental-Candidate Classification, 2021–2025
This notebook uses real_acct.txt as the base HCAD property table and merges the processed building and exemption components created in the previous notebooks. Because real_acct has one row per HCAD account, all merges are performed as left joins and the number of property rows must remain unchanged.
The 2021 rental-property methodology distinguishes between explicit rental properties and probable single-family rental candidates. Explicit rentals are identified from multifamily state classes, residential small-multifamily building types, apartment/manufactured-housing building types, and PGI categories. Probable single-family rentals are A1 properties with a residential building record, no observed homestead evidence, and an owner mailing address different from the property site address.
These are described as rental candidates, not verified rentals, because HCAD does not provide a complete rental-status field.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

In [2]:
project = Path(r"C:\Users\alys_\Documents\Research\Evictions")
hcad_root = project / "01_DATA_RAW" / "HCAD"
processed = project / "02_DATA_PROCESSED"
output_root = processed / "property_master"

output_root.mkdir(parents=True, exist_ok=True)

In [15]:
years = [2021, 2022, 2023, 2024, 2025]

wanted = [
    "acct",
    "yr",
    "mail_addr_1",
    "mail_city",
    "mail_state",
    "mail_zip",
    "site_addr_1",
    "site_addr_2",
    "site_addr_3",
    "state_class",
    "neighborhood_code",
    "neighborhood_grp",
    "yr_impr",
    "bld_ar",
    "land_ar",
    "acreage",
    "land_val",
    "bld_val",
    "x_features_val",
    "assessed_val",
    "tot_appr_val",
    "tot_mkt_val",
    "new_construction_val",
    "tot_rcn_val"
]

num_cols = [
    "yr_impr",
    "bld_ar",
    "land_ar",
    "acreage",
    "land_val",
    "bld_val",
    "x_features_val",
    "assessed_val",
    "tot_appr_val",
    "tot_mkt_val",
    "new_construction_val",
    "tot_rcn_val"
]

In [16]:
def load_real(year):
    path = hcad_root / f"pdata{year}" / "real_acct.txt"

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        encoding="latin1",
        usecols=lambda c: c.strip().lower() in wanted,
        low_memory=False
    )

    df.columns = df.columns.str.strip().str.lower()
    
    df["acct"] = (
    df["acct"]
    .astype("string")
    .str.strip()
     )
    
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
                .replace("", pd.NA)
            )

    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    return df

In [17]:
def make_master(year):
    df = load_real(year)

    start_rows = len(df)
    start_accounts = df["acct"].nunique()

    df["mail_norm"] = (
        df["mail_addr_1"]
        .str.upper()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )

    df["site_norm"] = (
        df["site_addr_1"]
        .str.upper()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )

    df["mail_site_different"] = (
        df["mail_norm"].notna()
        & df["site_norm"].notna()
        & df["mail_norm"].ne(df["site_norm"])
    )

    res = pd.read_parquet(
        processed / "building_res" / f"building_res_{year}.parquet"
    )

    other = pd.read_parquet(
        processed / "building_other" / f"building_other_{year}.parquet"
    )

    homestead = pd.read_parquet(
        processed / "homestead" / f"homestead_{year}.parquet"
    )[
        ["acct", "any_homestead_evidence"]
    ]

    fixtures = pd.read_parquet(
        processed / "fixtures" / f"fixtures_{year}.parquet"
    )
    
    for data in [res, other, homestead, fixtures]:
        data["acct"] = (
        data["acct"]
        .astype("string")
        .str.strip()
    )

    df = df.merge(
        res,
        on="acct",
        how="left",
        validate="one_to_one",
        suffixes=("", "_res")
    )

    df = df.merge(
        other,
        on="acct",
        how="left",
        validate="one_to_one",
        suffixes=("", "_other")
    )

    df = df.merge(
        homestead,
        on="acct",
        how="left",
        validate="one_to_one"
    )

    df = df.merge(
        fixtures,
        on="acct",
        how="left",
        validate="one_to_one",
        suffixes=("", "_fixture")
    )

    df["in_building_res"] = df["n_buildings_res"].notna()

    df["any_homestead_evidence"] = (
        df["any_homestead_evidence"]
        .fillna(False)
        .astype(bool)
    )

    bool_cols = [
        "small_multifamily",
        "apartment_type",
        "apartment_pgi",
        "manufactured_type",
        "manufactured_pgi"
    ]

    for col in bool_cols:
        df[col] = df[col].fillna(False).astype(bool)

    df["state_multifamily"] = (
        df["state_class"]
        .isin(["B1", "B2", "B3", "B4"])
    )

    df["explicit_rental"] = (
        df["state_multifamily"]
        | df["small_multifamily"]
        | df["apartment_type"]
        | df["apartment_pgi"]
        | df["manufactured_type"]
        | df["manufactured_pgi"]
    )

    df["probable_sf_rental"] = (
        df["state_class"].eq("A1")
        & df["in_building_res"]
        & ~df["any_homestead_evidence"]
        & df["mail_site_different"]
    )

    df["broader_rental"] = (
        df["explicit_rental"]
        | df["probable_sf_rental"]
    )

    overlap = (
        df["explicit_rental"]
        & df["probable_sf_rental"]
    ).sum()

    summary = {
        "year": year,
        "rows": len(df),
        "accounts": df["acct"].nunique(),
        "duplicate_accounts": df["acct"].duplicated().sum(),
        "explicit_rental": df["explicit_rental"].sum(),
        "probable_sf_rental": df["probable_sf_rental"].sum(),
        "overlap": overlap,
        "broader_rental": df["broader_rental"].sum(),
        "homestead": df["any_homestead_evidence"].sum(),
        "mail_site_different": df["mail_site_different"].sum()
    }

    assert len(df) == start_rows
    assert df["acct"].is_unique
    assert df["acct"].nunique() == start_accounts

    df.drop(
        columns=["mail_norm", "site_norm"]
    ).to_parquet(
        output_root / f"property_master_{year}.parquet",
        index=False
    )

    return summary

In [18]:
summary = []

for year in years:
    print("processing", year)

    s = make_master(year)

    summary.append(s)

    gc.collect()

processing 2021
processing 2022
processing 2023
processing 2024
processing 2025


In [21]:
df_summary_master = pd.DataFrame(summary)

display(df_summary_master)

,year,rows,accounts,duplicate_accounts,explicit_rental,probable_sf_rental,overlap,broader_rental,homestead,mail_site_different
0,2021,1507914,1507914,0,16571,184301,643,200229,824769,629516
1,2022,1531434,1531434,0,16519,188087,500,204106,852645,643281
2,2023,1555714,1555714,0,16830,191853,514,208169,868465,658520
3,2024,1582007,1582007,0,17421,195248,529,212140,878810,674614
4,2025,1601916,1601916,0,17402,201860,529,218733,881711,683027


In [19]:
real = load_real(2021)

res = pd.read_parquet(
    processed / "building_res" / "building_res_2021.parquet"
)

other = pd.read_parquet(
    processed / "building_other" / "building_other_2021.parquet"
)

homestead = pd.read_parquet(
    processed / "homestead" / "homestead_2021.parquet"
)

fixtures = pd.read_parquet(
    processed / "fixtures" / "fixtures_2021.parquet"
)

In [20]:
diagnostic = pd.DataFrame({
    "dataset": [
        "real_acct",
        "building_res",
        "building_other",
        "homestead",
        "fixtures"
    ],
    "rows": [
        len(real),
        len(res),
        len(other),
        len(homestead),
        len(fixtures)
    ],
    "acct_type": [
        str(real["acct"].dtype),
        str(res["acct"].dtype),
        str(other["acct"].dtype),
        str(homestead["acct"].dtype),
        str(fixtures["acct"].dtype)
    ],
    "first_acct": [
        real["acct"].iloc[0],
        res["acct"].iloc[0],
        other["acct"].iloc[0],
        homestead["acct"].iloc[0],
        fixtures["acct"].iloc[0]
    ],
    "acct_length": [
        real["acct"].astype(str).str.len().value_counts().to_dict(),
        res["acct"].astype(str).str.len().value_counts().to_dict(),
        other["acct"].astype(str).str.len().value_counts().to_dict(),
        homestead["acct"].astype(str).str.len().value_counts().to_dict(),
        fixtures["acct"].astype(str).str.len().value_counts().to_dict()
    ],
    "matches_real": [
        len(real),
        real["acct"].isin(res["acct"]).sum(),
        real["acct"].isin(other["acct"]).sum(),
        real["acct"].isin(homestead["acct"]).sum(),
        real["acct"].isin(fixtures["acct"]).sum()
    ]
})

display(diagnostic)

display(real[["acct"]].head(10))
display(res[["acct"]].head(10))
display(homestead[["acct"]].head(10))

,dataset,rows,acct_type,first_acct,acct_length,matches_real
0,real_acct,1507914,string,0010010000013,{13: 1507914},1507914
1,building_res,1197988,string,0020720000014,{13: 1197988},1197988
2,building_other,66731,str,0010020000016,{25: 66731},0
3,homestead,1507911,string,0010010000013,{13: 1507911},1507911
4,fixtures,1267694,string,0010020000016,{13: 1267694},1267694


,acct
0,0010010000013
1,0010020000001
2,0010020000003
3,0010020000004
4,0010020000013
5,0010020000015
6,0010020000016
7,0010020000023
8,0010020000024
9,0010030000001


,acct
0,0020720000014
1,0021440000001
2,0021440000003
3,0021440000008
4,0021480000001
5,0021480000002
6,0021520000004
7,0021650000007
8,0021650000011
9,0021660000004


,acct
0,0010010000013
1,0010020000001
2,0010020000003
3,0010020000004
4,0010020000013
5,0010020000015
6,0010020000016
7,0010020000023
8,0010020000024
9,0010030000001


## Results and decisions
real_acct was used as the base property population for each year, preserving exactly one row per HCAD account. The annual property population increased from 1,507,914 accounts in 2021 to 1,601,916 in 2025. All joins were performed as one-to-one left joins, and no accounts were added or removed.
HCAD account identifiers were standardized by stripping fixed-width whitespace before merging. This was necessary because some source files store the 13-character account identifier in wider fixed-width fields. Without normalization, visually identical account numbers failed to match across files.
Rental status was constructed as a candidate classification rather than a verified tenancy indicator. Explicit rental candidates were identified from multifamily state classes, small-multifamily building types, apartment or manufactured-housing building types, and PGI classifications. Probable single-family rentals were defined as A1 properties with a residential building record, no observed residential homestead exemption, and a mailing address different from the property site address.
The reconstructed 2021 procedure produced 16,571 explicit-rental candidates, 184,301 probable single-family rental candidates, and 200,229 broader rental candidates. The original 2021 proof of concept contained 200,438 broader rental candidates, a difference of 209 properties (approximately 0.10%). The original and reconstructed procedures produced the same 643-property overlap between explicit and probable single-family classifications. The small remaining discrepancy is retained rather than forcing agreement because the rebuilt pipeline uses the current raw HCAD sources and explicitly documented cleaning rules.
Across 2021–2025, the broader rental-candidate population increased from 200,229 to 218,733. These classifications should be interpreted as probable rental candidates because HCAD does not contain a complete direct rental-status field